# 01 · Data Preparation — from a WSI to a feature bag

**Introduction to Pathology MIL — Notebook 1 of 5**

Goal: turn one gigapixel whole-slide image (WSI) into the `(N × d)` tensor of patch
features that every MIL model consumes — a *bag*. We follow the standard CLAM-style
pipeline:

> **WSI → tissue segmentation → patching → frozen foundation encoder → bag tensor**

This notebook shows the **real TCGA/OpenSlide pipeline** *and* a **synthetic fallback**
so you can run the rest of the course with no slide download and no GPU.

## ⚙️ Colab setup

Run the cell below **first**. It writes the helper modules (`mil_models.py`, `mil_utils.py`) and regenerates any prerequisite data, so this notebook runs **standalone** in Google Colab — just choose *Runtime → Run all*. Colab already includes torch, numpy and matplotlib.

In [ ]:
# === Colab setup — RUN THIS CELL FIRST ===
# Makes the notebook self-contained: writes the helper modules used below.
# Colab already ships torch / numpy / matplotlib, so no pip install is needed.
import os, base64, gzip

_FILES = {
    'mil_models.py': 'H4sIADC+ImoC/91Z3W7byBW+11McKBdLeiXGEpBiYdQF4m22WcDxLmInN4aXGIkjcSpyRp0ZWnbSAL0qFnu5KFD0efomeZJ+M0NSpKx0nR8DRXlhUvNz5vx+55zxcDgclKJIS5XxwiTrW3r/t3/QS77gmss5pxffnxJbLjVfMqu0oYXSZHNOPzKbq0Itb/2Kuaq04clgcJFzw4lpjqFyzeZ2RJqzjM0KPqKDudKaz+0BiXJd8JJLy6xQ0lDJmbSBNmfzXMjliKSyA0aZVuuxkKCyLtjc72l5UFoshWQFfXv69IVboRJ6ds31LXlpqGRrQ4xmbElqMVgzO89pwZmtNDdEz+n9zz/Ty5/env37n9k7IqvIFCLjBKmENSNiMqNok0MRxNbrQsydFDHOAQFpBo6DNdfjQJdZC9YgDF1DQjBYGZ55TnOc6DiBdghPS91aOSJ+YzWj48Bw9Dym+nlEz48oOhtRhqH3f/+12YfBCVSTzgtmDDcxNM6NxUk42NxK8GTF3ElsCEqrrIBNS7biaTuZZswyw20yGML0C61KStNF5ZSSps4wSluIDu0H2wzqIcg0z3s/EimJGZJydzRZVHLuNsM0WPDdYDB4ROMv94DaYUInkKIQkh9576HHMPcNrZXC2JIokooKzrR0RmuH10yzkluuobkvy5I3CP2IcxAPEXTwQmUV3OXIGx2qfgEmx3BZx+bY8eNdGJYauQ9okpwwTJOnJBaC64Qu/JIgp3PQGXwpcXbzVDO+gO2EFDZNIyxajGD0NBPlEd624yb+N7xsOvKOdkTGavwcOsUNR5SLLOOyWfRkMq25do+p4ONRnLTnxO2UIw2rO4rO2d425IaQcPhuSwGMJX5RcPP+xFqrP2MCCjvnf6lcCLHCqe/UKyMK8jQsxiO38CU/fRWFzz8CHVRlo8Nk+iSO+5S3egz0a4qBUj+GGmUiXDdMZ7UuEYHBoy+4NEp3lJKDYst9J2p3nkc+gmvW281vsDlHUN5EkOz4ME6uWVEBkMSiq6rjoEYCKHO3HKqt13u6tRRbqgEeGr62skdvkkoaaJa/4dFh/CFWA7eAlm+3JB9RJQVUUtKwRbchGUWZ2kh4EGdlA25AezBtADcFbZRema2TAObAVVDkoiqKKHqemJyt+eXh1QhWnCSHiN3OGCxxLeb8+HkSPrYMBeTdQdC37x4AXiYJPT1xic0lw1b4sQvFzCe86HtnFw6kLJIRTQ8n30AUpEhMXzMtkM0eBF/+5E542vCzD2ZYKpwTqAXMcpOKiDY/XdAlWSbz6DXlqYjp/S//IiOWpRJZ9CoMXVF8f1jpw8X0ye9GPksjDo9oUSjmRn1A3gtFvL++/kgIuHDi9CGgZmEXBV59JOXzWjH3Ir7ZiyyTDyJKvosovfDLQ8YPXHVgtgnqTRR0FeUxHdTS4TvewZtJjLiVCFtWiDfwR4Oqi5seucBE7SQRg/1qZNkDXyBnqtIlnwkpFFdN2tqNSoYwDE7qI2efb3rv7VRKTVb+ek/q+6wkdyehBbBI73juR7jowyWqGiJ3Yrvxpg7j/3MJ7oM5rvVZx73z19/YP+mnx4jBwfM4geOFpDeiFedr93mhKx53E1Z9dAelN1wsc+dp2H6P/Bj/Vh7cTTtJk04n8QOln2niW5rx+YlPQMuduPkaDmyA6HM+nhcVyn/toohVN6IQDN1PoYwBFTzRadUmqTPf+NCJUCXP0MwU9EyifeJh9/RwOnmYmthJkp6f7AME/z7H8QUfzzQEyv1qJN8sQ+cm6aARtAMNByFUr4XLxbNbj0cWTeLKNWye4kxZq0o34FSWYZnv0VBjRWvDq0yNCzbjReEmlBFWXHOUINK1uPiMQ9kNzKwKC948yZYPp1vMLasCaR7B5btE1C56jRjyvGzthMIooTNlUXBb30YKQ2c/eHqrsavoDK3RtWo155ANRnDGdiSGW7MO3Sab48/Wvi0zaORWSavMLwiXgztxuh8/EZepYa6Zbwa/+QhIbfZiW/PZX9Ay6wCu+f7/geUt4ii0dzMhe8bt0IEaw0+CQI/BMKlFGNg5u9mcbjebwEUIvVNhbHR5l6lp7K8qUtfHIQ6XPNqyedXJH2l7gIuDpqqBNkaBnVRkNx0HgFdehMhsAtFdZdQx2ITeaBuvndkmGpMGJtxz5lunplNoh1eusxQy6jkVAI8eP6ap66vO6A+oXnwr1ck0mFjR72ly1PP2Gu3zRPJN+oZrZaIo7mc2llwLvonGHVoAoLauwvfKFVWrGPYA0nZcFpL2l433r4Ma23VzBpNhsdfTVedIppfcp7TeOvcJdzLRCmWdvV3z4zBWKHepVvdWedNb7Yn07hO2Bi3ci97VhzrSfZ552frMVZRfulfcZODpyjnlbgr+LpmDE5MiFhG4t1GTkmtV3LfQGZHH/wBZfwVCSwdB7vVJJdBn10CfUwT1q6BPKHf29f37yguf+Jqb18iCgxFt3MWV1QzwLpdxDVEhkW9dGbQ8WvjY3R9UiERvEZfrJELEmaIflV0qXqQdJAoYBHtGnlB83+pt2CMzPNoe5Oo650mzShRZuB6PJCvD5dk9kurBwWpTu5Pb5lAYL4TNxuXEQS22n+pfnuHtGqPhu6NdGZrbxSaptceGa73jQGMfbXZT38t9MmVQ6BN2N1RsVopieIdaaAD30PI6ucse5kvHn3unZraHv6aG/K804YcA+NfuNu2Z1kpHi2ElV1JtZP3/gK/eujPffQVJ/gPZBRiT9xgAAA==',
    'mil_utils.py': 'H4sIADC+ImoC/71a247bxhm+11MMFKQh1xItrbtBqoQBNm4SGIidRbxuUggCM0uOpLEokpmhdlc2DOS21y3QJ+iT9E3yJP3+OfAg7dpNYVcXXmk4/+mb/zj0cDgcbGWe7GqZ66jas99+/Qd7vuZKZGwt8koozZalYvVasAter8u8XO3Z0yffsbTcKS1YUdbiqiw3OhoMLlR5LTOhZwPGTtiWb0Si9wVIa5kmGa+5FjUzEpTgudRYZs0GthS83inBTq74Sp8wXTJxLdS+kQCmjKldoZkosnFdjvGH3UjotKtZVt4UeckzWazY5eNvz9lDxtm3Fy8i9lRuZaqNAcGzf/8zC1ktCl3CLm45QhWYuCugoCwLME/LTCh2U+7yjFWqzHapGBlBYFnlvKiBzbDebUs1ZFquCp5HxuAKDERRJ7pW+LaUIks2yxJMyOJc8A1fibHmS8Ee/4XpKpd1DTmB+cau9p4+tNzARBZJWQiYIq55vuO1IH0Nt60s5BZ600GYjW4Ty8uy0pZBWm6rXS2SragVIdB8iMH5ix++fzxiVxz2pLCHp+lO8XQ/wpOLH/BE1ClO9BKotSe0EoWAaXCHXNS6PXo2OWUchzE583ixE4LVGQH2W0Ca6wFtck+sVgxSr4WO2JMCpDyFFMH25c6hrwTwxsrPd/vSzwNzKuQvTNxWpaKTuWo9hk2mbKnKLayrdwDnx+dPgM0QLm9Wk2S5I49LEia3RA0jQGrcQA/cUrHbIio4rK0GH7Fn319+PWNAIF0zqR0ZhOb8lcz3TBYa/m98zVgOb3xojmW5K1LDlty6XvOa9oAfGTKuYKbBz5+UD7ybUm3AE25nJI6XSgj457VUZbGFp+CAwGP8/j7g9rw57aEPSMJ3aLW+Emt+LVguN+IwcMY+cBCO8Lv3rFgmlvcklKBIXODo+LPJCHglmdzGZ9PTEdNCZPFkNGD3fxBJRJ+uhY5PJ6Df8ttm4U+TtxNXpU6Wxm3LIp5EZyOXD+Jp9Gk4M5TkbfT3BwEscf6cUeJj5ZJlMq31iFGE47SZzuE6s0baa59NZDayz8y3nF+JfORzpUZKG2VsicxXPzoNR4j5UmVm9TR8MzDMztkJ1JS1vBYnlhF2FTW8k5TRyCI4RWcDqeUzm8OAZYrfFC6MDEO9lkvy+W/5TmvJCxaQv/vEaAEIP2eFWHESamVqLxTBWYiIXa4pfLRhKG4hHdFDbCihca0RdKTPzKzB/yjYvBVMLpfst7/9a0rhVlP66j6NeqgrlIMYoRspxFe5jeBGfJfXCdYDco7Q7PoIQCzlLbS32xwGOCElDC5DikIfDrpCSjJ0Zhe8TUEGOEakTcZVlhSlAq6BdcUw4rreVyIYunMahgfUD42KuQRwq4hog+ZZaA+RfB1C5gvziwpyRSpB3ZXoBEDY+o9xFJDIog5IN2tZELIvel4bNgRF4g7K2gI6sUISCqYj9igkjKa//fr3U3+a5LFOasOB1NJdtezejlL0edZRqhHSicJeBIZhj/ane4CGv7u4D+9Am5S/4ulmpShZsVrCv0SPr1x6vGI2nR1F/EfWu9twKdOdplixbuIjJTgDQtOzj80T67fhEa8isUQxmUngEhbPUK7Jrl0hl3T8k2iCXDKJpmdheMxCZrcOh3RdomCS9Y7tyJfM+Buea3FM+9Mc1Av2IHaBSo2G97WjzfbJluuNDaJXQpWapGWEcIwSS4He7rK8Y3apDvBF7Rez/4n9oH8OS5QAdjr+s8XcpjuETQ30S6qTKyUzFpAjrhGtW17pPgdTnq3/QWAqZE5/9S8KZ3AI9R6ueKutapm8RgtDe7l17mfhyDDrk7j0a0jgo+kmmN+Cy14vRozfSh1PW/eEDsY5T9jp2ac9LhTsEa8q9LcBFYmgLQXxcnjxuppNHmVvhm1Z6Kwmz19rPHpb0XK5IT4oJfFPvnzE9s87mLSnF7dfHYTKFDtjyKjjX++/W/mu21WncB89RrslbUcyY01rfXF++eTrZ5eArOnNadlA8AE6lfsmgcAiUiT0Q8dnvkNpe4W/SoG+N7DNP8JpxGAPfQlZee27BPgv/IJSZSHMopklaPThXnT0eyugo2PjL10qDBpen2if9TUNhia32T1roVwE+AoEWa/fNEVKIpapIAh00TQ5CANApyB4sgiNnNMsyObD1uGHiJzXQyNtOGN4ZL9idQhQsDRfvAmP2M0PeCzmZvfCR5V0SksTrNSQBU0JHTQR4gKZK8X3wbzhXS0aLdpKTKwWrlibw6VqPbcbkm5NNA/Dto6DEz2GHCT/X3YisKI7GK1URcxIwlxajtKR3NABBF7ZmJiF88li0dCaYrneLZe5CMAn7NXqlyNSu38+tKmfqY3C85fsY++2DYogbixWbHlsZcuInNihTS275blctPocO7xpd0a+5fmvPYo+gWeBun7gCUTbKENlqZUcHniHqQEmHI+V+wCp7Kkd/j5ANjq4CAj2SY0CjbKUYFq/anOPuxNwFwB3XA34bwyTIKadiF2gcIzNhIxUVDK9yQWnvC+qJv9YYS6StI0lu9aUQlTC0O0lhQ730prvCUxP53timMZ3qkwDSO96LbDEII9e7Me1rAuxZy+Q9uED5r7J3BjAH8aYYxVKh+n5N2jeED5ef/BM+C5NdFoiv0WdFhmtM6VvzDaui9hHmFWCMLR9XDBlY7YP3dqg014aUgrRCYMXOw741XdcVzeNlcGw4MWwZYKSLJTPSCtdomWpRmyDUhAPEQorQUud/ZqyRjU3ZJ2EQMYm2t5aOGYmZtGJ5qIIqpA9YNMDuD3xS0b6v/ODccpi66AlyBF2AJ1RC17phuHNWuYCbL8wsvVh6qG+8GVvxRJsSMeWyNyd6GpulhcEK368XBw3nBvqe6d9xDt4zF/OPAsWvDQyHlhZIXvITqNJj5LAMA/72FpQBebXfWABvQdMs9sdD7Xz3YPpgWnuSleFY1+X8AS5olnRusTgwHkCy3hvZpmFdUV4pXXAExbYL94o+ifwz4xb9qKrUkfRdYcfjjuFBXLZ/tDr6soSpLst6bPv1KH+Exs/bT3H/I3nIH9ImzAxyS124fcDUI5gQ8f6ZicNVi4u79hCwsoiRemguSGYo17S1JQuMMpU92yZYktl9gzuDFUQoEZUrwLaZLiFDsZKmThzaYx9GVPiPM58BiCTQQJLEdP5/IEFPn3S715eqYsjiskBxaRPsXy3jEOKd8s40EqLQnePwZ5U0RyDruwxFf550T9JPzjQ1NNAbZK8DZ/AJfx+BQtHnb1w2Wavcd979/oKl6CuOZLAGPDA6Gnjo8PaVT+/1dhWF6EzhYLdFTYyp0NIPOluStaellZogIQUTAdp5wHJtZThh+gx3vraAI3TlRmrtXxFFXtqbzDef0OC9lDdcJUF5rWAHUG1u9uMn5WFaHuSF/ZGhKV072KuLMwdHDezHrsoy5xsecjOv7J/H393/jR5/lXTftRqP7u3TNFmOlZR1boz1fj3Xc31Yg5Zh4FvVA96mpt/Q3elSUzZJYL8a6VKNXsLOR00odK88PGwuPH5qPkcsZMRE1WZrnX8aALpKj4V4z+O2E0WT8X47HBwJzuSGyFX6zqeRI8w5udca7eiDeAQJq4xUsbDtNoNR26Qwu/PIFWoq1L7GyVriXs5Yl5LHC1ERRH5Fx7wLa7ZN2aPMSuqy8DKskiVgCl2dPgut9F5xrc/WgyiCt0J2la6IgyNobmCmUbxJBMp38c3boJFj9Uzi+6Ei7JmZF2Lfn+Ll2tfCAa9h75q2x3uLrGBqWvBlQC8aBdH9hu1mvS+kEsybDyNcEIWYts/rNGHlmrfv80Vnetce7KdiutwIz8I7uwH3WhfCbXd2ddX7QVCf+Qzg6Oh7HdHVKPI3TBg9oc/ctAGJ3oJkJh2n4Z0f3E0XISHh9ptBnoQz9sJfnEnmM0VVbmSdAgJPP0WxoDRW7NGy/eQjyb9v4lMwkgwBqoS6nv2e+9Nce/ww8Pb4aHPBQkxNK8DjFrHPaYTaP486IYe+itDMj/g1QccIWDuQZOV4lmAloT2RHR7bUzHAu3Qtag6rkAZPG5eEvfTR5Mz7gHbuaOffbF9PjRFtoskEOg+YF82Tt8HwK9Cm+7+zzuRQRdEmxm7jjJR83QdhBEyDv2bU+YLB+z3f8ixN7CUTsXGipGUmC4ijGQttkgfbz73Mdn28cf30mZLb0og220G7O+sFHVGyyGzqZi9FtVscpq9scdhZun4NaHwiUHhk8UserTE06tzdBv2QbcBcc+HPdCNNugZfTo+QBuTwcYnvw7Cd2Y+Cwz9F4mkg05L1eu/nAc533D16W7/6vqWKx6WiWm39NurhlWKGLuTpzv3SvevfuzEbguLj4vZcVaDIv+XnNbJTMm7spLpZfpeo32kWfG6XName3Rc6dX1NJyjZmB2I88Njt5KNK8GDlLeFtocXfUYPJsK2T2Z45Zk1N527qk9bX55Dn7j4D/FS2hjNiQAAA==',
}
for _name, _b in _FILES.items():
    with open(_name, 'w') as _f:
        _f.write(gzip.decompress(base64.b64decode(_b)).decode('utf-8'))
print('helper modules written:', list(_FILES))

## 0 · Environment

Real pipeline needs: `openslide-python` (+ the OpenSlide C library), `opencv-python`,
`torch`, `timm`, `huggingface_hub`, and access to a foundation encoder
(e.g. `MahmoodLab/UNI2-h` or `MahmoodLab/CONCH`, which are gated — request access on
the Hugging Face Hub and `huggingface-cli login`).

```bash
pip install openslide-python opencv-python torch timm huggingface_hub h5py numpy matplotlib
# system: apt-get install openslide-tools   (or: brew install openslide)
```

In [ ]:
import numpy as np, os
import matplotlib.pyplot as plt

# Toggle: set to True if you have OpenSlide + a real .svs slide + GPU access.
USE_REAL_WSI = False

# Real pipeline config (edit paths to your environment)
WSI_PATH   = "/data/TCGA/TCGA-XX-XXXX.svs"   # a TCGA diagnostic slide (GDC portal)
ENCODER_ID = "MahmoodLab/UNI2-h"             # gated; or "MahmoodLab/CONCH"
TARGET_MPP = 0.5      # 20x  (microns per pixel)
PATCH_PX   = 256      # tile size at the target magnification
OUT_DIR    = "bags"; os.makedirs(OUT_DIR, exist_ok=True)

## 1 · Open the WSI and read its pyramid

A WSI is a multi-resolution pyramid. **Always read the true microns-per-pixel (mpp)** —
"20×" differs between scanners. We compute the pyramid level whose downsample gets us
closest to `TARGET_MPP`.

In [ ]:
def open_wsi(path):
    import openslide
    slide = openslide.OpenSlide(path)
    mpp_x = float(slide.properties.get(openslide.PROPERTY_NAME_MPP_X, "nan"))
    print("levels:", slide.level_count, "| dims L0:", slide.dimensions,
          "| downsamples:", [round(d, 1) for d in slide.level_downsamples])
    print("native mpp_x:", mpp_x)
    return slide, mpp_x

if USE_REAL_WSI:
    slide, base_mpp = open_wsi(WSI_PATH)
    # patch size in level-0 pixels so that the *physical* tile == PATCH_PX @ TARGET_MPP
    patch_l0 = int(round(PATCH_PX * (TARGET_MPP / base_mpp)))
    print("patch size in level-0 px:", patch_l0)

## 2 · Tissue segmentation (skip the glass)

CLAM recipe: downsample → HSV → Otsu-threshold the **saturation** channel → median blur
+ morphological close → contours. This bounds where we sample patches.

In [ ]:
def segment_tissue(slide, seg_downsample=64, sthresh=8, mthresh=7, close=4, min_area_frac=1e-3):
    import cv2, openslide
    # pick a level near the requested downsample
    level = min(range(slide.level_count),
                key=lambda l: abs(slide.level_downsamples[l] - seg_downsample))
    img = np.array(slide.read_region((0, 0), level, slide.level_dimensions[level]))[..., :3]
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    sat = cv2.medianBlur(hsv[..., 1], mthresh)
    _, mask = cv2.threshold(sat, sthresh, 255, cv2.THRESH_OTSU + cv2.THRESH_BINARY)
    kernel = np.ones((close, close), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    min_area = min_area_frac * mask.size
    contours = [c for c in contours if cv2.contourArea(c) > min_area]
    scale = slide.level_downsamples[level]   # map contour coords back to level 0
    return mask, contours, scale

if USE_REAL_WSI:
    mask, contours, seg_scale = segment_tissue(slide)
    print("tissue contours kept:", len(contours))
    plt.imshow(mask, cmap="gray"); plt.title("tissue mask"); plt.axis("off"); plt.show()

## 3 · Patching — tile the tissue into instances

We lay a non-overlapping grid over the slide and keep tiles whose centre falls inside a
tissue contour. **Coords-first**: store `(x, y)` level-0 coordinates, lazy-load pixels at
encode time (CLAM's fast, disk-light trick).

In [ ]:
def grid_coords(slide, contours, seg_scale, patch_l0):
    import cv2
    W, H = slide.dimensions
    coords = []
    for y in range(0, H - patch_l0, patch_l0):
        for x in range(0, W - patch_l0, patch_l0):
            cx, cy = (x + patch_l0 / 2) / seg_scale, (y + patch_l0 / 2) / seg_scale
            if any(cv2.pointPolygonTest(c, (cx, cy), False) >= 0 for c in contours):
                coords.append((x, y))
    return np.array(coords, dtype=np.int64)

if USE_REAL_WSI:
    coords = grid_coords(slide, contours, seg_scale, patch_l0)
    print("patches in tissue:", len(coords))

## 4 · Feature extraction — patches → embeddings (the expensive step)

Each tile is read at level 0, resized to `PATCH_PX`, normalised, and pushed through a
**frozen** foundation encoder. This is the one GPU-bound pass; cache the result.
Embedding dim `d` depends on the encoder (UNI2 = 1536, Virchow2 = 2560, CONCH v1.5 = 768).

In [ ]:
def build_encoder(encoder_id):
    import torch, timm
    from huggingface_hub import login  # ensure you've logged in & accepted the licence
    # UNI / UNI2 load via timm; see the model card for the exact init args.
    model = timm.create_model(f"hf-hub:{encoder_id}", pretrained=True,
                              num_classes=0, dynamic_img_size=True)
    model.eval().requires_grad_(False)
    return model

def extract_features(slide, coords, model, patch_l0, patch_px=256, batch=128, device="cuda"):
    import torch
    from torchvision import transforms
    tfm = transforms.Compose([
        transforms.ToTensor(),
        transforms.Resize(patch_px, antialias=True),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    model.to(device)
    feats = []
    with torch.no_grad():
        for i in range(0, len(coords), batch):
            tiles = []
            for (x, y) in coords[i:i + batch]:
                im = slide.read_region((int(x), int(y)), 0, (patch_l0, patch_l0)).convert("RGB")
                tiles.append(tfm(im))
            out = model(torch.stack(tiles).to(device))      # (B, d)
            feats.append(out.float().cpu())
    return torch.cat(feats)                                  # (N, d)

if USE_REAL_WSI:
    import torch
    enc = build_encoder(ENCODER_ID)
    bag = extract_features(slide, coords, enc, patch_l0)
    print("bag tensor:", tuple(bag.shape))
    slide_id = os.path.splitext(os.path.basename(WSI_PATH))[0]
    torch.save({"features": bag, "coords": torch.from_numpy(coords)},
               os.path.join(OUT_DIR, f"{slide_id}.pt"))
    print("saved", f"{slide_id}.pt")

## 5 · Synthetic fallback — make bags without a slide

So the rest of the course is runnable anywhere, `mil_utils.make_synthetic_dataset`
generates `(N × d)` bags that behave like real encoder output: most patches are
background, and **positive** slides contain a small planted "tumor" focus — exactly the
MIL assumption (bag positive ⇔ ≥1 positive instance).

In [ ]:
from mil_utils import make_synthetic_dataset

data, tumor_dir = make_synthetic_dataset(n_patients=80, in_dim=512, seed=1)
print(f"{len(data)} slides | positives: {sum(d['label'] for d in data)}")
d0 = next(d for d in data if d['label'] == 1)
print("example positive bag:", d0['features'].shape,
      "| tumor patches:", int(d0['tumor_mask'].sum()), "/", len(d0['tumor_mask']))

# persist for the next notebooks
import pickle
with open("synthetic_bags.pkl", "wb") as f:
    pickle.dump(data, f)
print("saved synthetic_bags.pkl")

In [ ]:
# Visualise a bag: project features to 2-D and colour the planted tumor patches
X = d0['features']; m = d0['tumor_mask']
Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
pc = Xc @ Vt[:2].T
plt.figure(figsize=(5, 4))
plt.scatter(pc[~m, 0], pc[~m, 1], s=8, alpha=.4, label="benign")
plt.scatter(pc[m, 0],  pc[m, 1],  s=18, c="crimson", label="tumor (planted)")
plt.legend(); plt.title("One bag in feature space (PCA)"); plt.xlabel("PC1"); plt.ylabel("PC2")
plt.tight_layout(); plt.show()

### ✅ Takeaways
- A bag is `(N × d)`: `N` tissue patches, each a `d`-dim embedding from a **frozen** encoder.
- Segmentation + coords-first patching make this fast and disk-light.
- Feature extraction is the only GPU-heavy step — **cache it** and every later notebook reuses it.

**Next:** `02 · Training` — train an attention MIL aggregator on these bags.